In [ ]:
# Create vanilla NN: feature selection ---> split dataset ---> modeling ---> evaluate model
# Imports
import os
import sys
import numpy as np
from huggingface_hub import login
from tqdm import tqdm
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from pricer.evaluator import evaluate
import csv
from sklearn.feature_extraction.text import CountVectorizer
# Torch dependencies
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

In [2]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Download the batches dataset from Hugging Face 
username = "omarshaarawy11"
dataset = f"{username}/best_deal_03_batches"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

Loaded 21,091 items


In [4]:
# Invistigate the dataset
columns = ['title', 
'category', 
'price',
'full',
'weight',
'summary',
'prompt',
'id'
]
for i in columns:
    # Combine item with its attrs
    attr = getattr(items[0], i)
    print(attr)

Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY
all beauty
6.99
None
0.0
Title: Lurrose 100pcs Full Cover Fake Toenails  
Category: Nail Art & Accessories  
Brand: Lurrose  
Description: High-quality, durable artificial toenails perfect for DIY nail art and manicures.  
Details: Features 100 lightweight, eco-friendly ABS nails that are easy to trim, shape, and fit comfortably to natural toenails, suitable for various styles and perfect as a gift.
None
None


In [5]:
# Feature selection
# Convert prices (target) into numpy array
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [6]:
# Split dataset
# We add new useful feature
prices_train = np.array([float(item.price) for item in train])
prices_test = np.array([float(item.price) for item in test])
# Apply NLP 
# Add corpus
# Then we come up with the most k-words in corpus and treeat them as seperated features
documents_train = [item.summary for item in train]
documents_test = [item.summary for item in test]

In [7]:
# Encoding
# Apply hashing vectorizer to form new X_train and X_test
np.random.seed(40)
# Ignore stop English words, We need the important ones
vectorizer =  CountVectorizer(max_features=450, stop_words='english', binary=True)
X_train = vectorizer.fit_transform(documents_train)
X_test = vectorizer.transform(documents_test)

In [8]:
X_train

<16800x450 sparse matrix of type '<class 'numpy.int64'>'
	with 369965 stored elements in Compressed Sparse Row format>

In [9]:
X_test

<2191x450 sparse matrix of type '<class 'numpy.int64'>'
	with 42522 stored elements in Compressed Sparse Row format>

In [10]:
# Modeling
# We have 8 layers using Pytorch
# Not enough number to be deep NN it's vanilla NN
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        # Construct layers
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Forward pass
        # Here we get the final output after passing through layers calling previous layer
        # Activation function 
        # The first output from here
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        # No activation function for last layer
        output8 = self.layer8(output7)
        return output8

In [11]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train.toarray())
y_train = prices_train
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)

# Split the data into training and validation sets ---> extarct validation set from training set
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [12]:
# Calculate paramters number
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 86,849


In [13]:
# Define loss function and optimizer
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

100%|██████████| 260/260 [00:22<00:00, 11.53it/s]


Epoch [1/2], Train Loss: 18781.281, Val Loss: 5470.128


100%|██████████| 260/260 [00:08<00:00, 29.25it/s]

Epoch [2/2], Train Loss: 1690.535, Val Loss: 5121.787


In [14]:
# Test dataset
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    # Return zero in case we have negative result    
    return max(0, result)

In [15]:
# Test single datapoint
neural_network(items[0])

17.53412628173828

In [16]:
# Evaluation
evaluate(neural_network, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$8 $48 $4 $3 $0 $4 $2 $17 $1 $3 $11 $6 $75 $10 $13 $5 $15 $14 $299 $240 $8 $25 $39 $19 $6 $10 $9 $38 $1 $1 $4 $13 $44 $711 $5 $22 $18 $1 $36 $1 $40 $82 $2 $37 $32 $24 $20 $11 $4 $1 $39 $12 $17 $2 $2 $17 $23 $20 $13 $9 $51 $15 $16 $15 $58 $125 $8 $0 $11 $13 $4 $4 $33 $41 $18 $0 $7 $21 $33 $223 $34 $0 $11 $128 $37 $18 $5 $43 $4 $1 $15 $3 $7 $5 $5 $5 $3 $19 $28 $1 $5 $1 $14 $8 $1 $41 $6 $42 $26 $3 $25 $0 $50 $67 $14 $54 $6 $86 $3 $44 $41 $3 $16 $8 $98 $31 $22 $12 $13 $68 $8 $6 $229 $93 $8 $0 $1 $19 $3 $4 $2 $24 $2 $7 $1 $8 $7 $26 $7 $98 $32 $27 $16 $9 $85 $14 $29 $3 $5 $4 $14 $42 $4 $0 $66 $250 $31 $74 $46 $7 $1 $95 $30 $124 $9 $82 $25 $10 $50 $5 $13 $1 $4 $11 $79 $3 $7 $24 $4 $5 $9 $5 $29 $7 $15 $44 $28 $1 $15 $15 $9 $25 $12 $100 $1 $14 $9 $32 $18 $14 $1 $12 $31 $5 $2 $109 $20 $9 $143 $22 $17 $1 $24 $45 $5 $49 $36 $20 $5 $9 $33 $12 $1 $6 $14 $31 $94 $11 $3 $3 $8 $6 $10 $2 $22 $7 $11 $15 $48 $289 $41 $4 $19 $2 $73 $486 $136 $0 $33 $33 $2 

$3 $14 $11 $4 $42 $8 $2 $48 $89 $29 $20 $12 $6 $9 $4 $11 $6 $35 $16 $14 $7 $16 $6 $64 $26 $65 $9 $34 $67 $81 $8 $9 $16 $22 $21 $4 $79 $12 $291 $0 $114 $8 $118 $11 $8 $1 $5 $174 $19 $7 $162 $2 $28 $65 $6 $84 $48 $19 $5 $16 $14 $20 $5 $10 $8 $1 $6 $7 $2 $28 $17 $67 $3 $8 $31 $12 $17 $12 $8 $9 $20 $84 $14 $15 $8 $37 $40 $5 $18 $4 $45 $4 $545 $52 $14 $6 $29 $30 $74 $19 $30 $9 $11 $19 $3 $21 $18 $39 $14 $47 $21 $13 $13 $76 $8 $2 $33 $6 $25 $15 $8 $6 $8 $22 $20 $4 $5 $18 $221 $12 $14 $3 $1 $18 $14 $2 $63 $47 $42 $7 $16 $52 $16 $23 $138 $0 $12 $2 $71 $4 $0 $45 $4 $7 $24 $9 $13 $144 $4 $154 $38 $14 $10 $58 $5 $7 $15 $14 $4 $18 $79 $9 $14 $11 $2 $9 $0 $2 $317 $0 $62 $12 $2 $11 $13 $0 $42 $22 $9 $10 $0 $28 $11 $48 $19 $20 $2 $28 $13 $1 $20 $3 $22 $50 $4 $20 $8 $7 $8 $14 $0 $32 $21 $15 $16 $20 $3 $12 $8 $13 $7 $137 $23 $4 $18 $656 $17 $6 $5 $5 $31 $18 $17 $27 $26 $3 $63 $61 $34 $131 $2 $41 $87 $65 $32 $6 $34 $7 $42 $4 $5 $15 $22 $24 $115 $11 $0 $6 $65 $136 $5 $4 $5 $4 $47 $7 $26 $4 $0 $5 $13 $6 $